# MindChat Standalone Kaggle Inference Server

Chỉ load `X-D-Lab/MindChat-7B` và expose `/v1/generate` qua ngrok. Notebook này chỉ chạy model baseline độc lập và expose HTTP API.


In [1]:
# 1. Install dependencies. Run this cell once, then restart runtime if packages changed.
# MindChat-7B uses custom InternLM model code. We avoid its broken custom tokenizer by loading tokenizer.model with LlamaTokenizer first.
!pip install -q -U "transformers==4.30.2" "huggingface_hub==0.23.5" "accelerate>=0.20.3" bitsandbytes pyngrok fastapi uvicorn nest_asyncio sentencepiece protobuf tiktoken einops


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [2]:
# 2. Runtime setup
import os

# Optional: choose GPUs before importing torch. Example for dual GPU: MINDCHAT_VISIBLE_DEVICES=0,1
visible_devices = os.getenv("MINDCHAT_VISIBLE_DEVICES", "").strip()
if visible_devices:
    os.environ["CUDA_VISIBLE_DEVICES"] = visible_devices
    print(f"CUDA_VISIBLE_DEVICES={visible_devices}")

# Set CUDA allocator knobs before importing torch. This matters on Kaggle T4.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("PYTORCH_ALLOC_CONF", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

import gc
import re
import torch
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

# Kaggle Secrets are not always injected into os.environ automatically.
def get_secret_or_env(name: str, default: str = "") -> str:
    value = os.getenv(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name).strip()
    except Exception:
        return default

def gpu_mem_info(index: int):
    try:
        free, total = torch.cuda.mem_get_info(index)
    except TypeError:
        with torch.cuda.device(index):
            free, total = torch.cuda.mem_get_info()
    return free / 1024**3, total / 1024**3

# Optional Hugging Face token if Kaggle download hits rate/auth issues.
HF_TOKEN = get_secret_or_env("HF_TOKEN") or get_secret_or_env("HUGGINGFACE_TOKEN")
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Hugging Face token loaded.")
    except Exception as exc:
        print("WARNING: Hugging Face login failed:", repr(exc))

if not torch.cuda.is_available():
    print("WARNING: GPU is not enabled. Turn on GPU in Kaggle Accelerator settings.")
else:
    GPU_COUNT = torch.cuda.device_count()
    print(f"CUDA GPUs visible: {GPU_COUNT}")
    for gpu_idx in range(GPU_COUNT):
        free_gb, total_gb = gpu_mem_info(gpu_idx)
        print(f"GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)} | free/total {free_gb:.2f}GB / {total_gb:.2f}GB")


CUDA GPUs visible: 2
GPU 0: Tesla T4 | free/total 14.46GB / 14.56GB
GPU 1: Tesla T4 | free/total 14.46GB / 14.56GB


In [3]:
# 3. Load MindChat model only
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig, LlamaTokenizer

MODEL_NAME = os.getenv("MINDCHAT_MODEL_NAME", "X-D-Lab/MindChat-7B")
# One value applies to all GPUs: 12GiB. Multiple values map by GPU index: 12GiB,12GiB.
GPU_MEMORY_LIMIT = os.getenv("MINDCHAT_GPU_MEMORY", "12GiB")
CPU_MEMORY_LIMIT = os.getenv("MINDCHAT_CPU_MEMORY", "32GiB")
ALLOW_FP16_FALLBACK = os.getenv("MINDCHAT_ALLOW_FP16_FALLBACK", "0").strip() == "1"
print(f"Loading {MODEL_NAME}...")

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def build_max_memory_map(gpu_memory_limit: str, cpu_memory_limit: str):
    if not torch.cuda.is_available():
        return None
    gpu_count = torch.cuda.device_count()
    values = [item.strip() for item in gpu_memory_limit.split(",") if item.strip()]
    if not values:
        values = ["12GiB"]
    if len(values) == 1:
        per_gpu = {idx: values[0] for idx in range(gpu_count)}
    elif len(values) >= gpu_count:
        per_gpu = {idx: values[idx] for idx in range(gpu_count)}
    else:
        raise ValueError(
            f"MINDCHAT_GPU_MEMORY has {len(values)} value(s) for {gpu_count} GPU(s). "
            "Use one shared value like 12GiB or one value per GPU like 12GiB,12GiB."
        )
    per_gpu["cpu"] = cpu_memory_limit
    return per_gpu

# If this cell is rerun after a failed load, release any previous partial model reference.
try:
    del model
except NameError:
    pass
cleanup_cuda()
MAX_MEMORY_MAP = build_max_memory_map(GPU_MEMORY_LIMIT, CPU_MEMORY_LIMIT)
if torch.cuda.is_available():
    print(f"CUDA GPUs visible at load: {torch.cuda.device_count()}")
    for gpu_idx in range(torch.cuda.device_count()):
        free_gb, total_gb = gpu_mem_info(gpu_idx)
        print(f"GPU {gpu_idx} free/total at load start: {free_gb:.2f}GB / {total_gb:.2f}GB")
    print("Accelerate max_memory map:", MAX_MEMORY_MAP)

def load_mindchat_tokenizer(model_name: str):
    """Load tokenizer without depending on MindChat custom tokenizer init order.

    The remote InternLM tokenizer calls get_vocab() before sp_model exists in some
    Kaggle/Transformers runtimes, causing: TypeError: property object cannot be
    interpreted as an integer. MindChat ships a SentencePiece tokenizer.model, so
    LlamaTokenizer is the safest first path for standalone baseline inference.
    """
    auth_kwargs = {"use_auth_token": HF_TOKEN} if HF_TOKEN else {}
    errors = []
    attempts = [
        (
            "LlamaTokenizer tokenizer.model",
            LlamaTokenizer,
            {"use_fast": False, **auth_kwargs},
        ),
        (
            "AutoTokenizer remote code",
            AutoTokenizer,
            {"trust_remote_code": True, "use_fast": False, **auth_kwargs},
        ),
    ]
    for label, cls, kwargs in attempts:
        try:
            print("Trying tokenizer:", label)
            tok = cls.from_pretrained(model_name, **kwargs)
            if tok.pad_token_id is None:
                tok.pad_token = tok.eos_token
            print("Tokenizer loaded with:", label)
            return tok
        except Exception as exc:
            errors.append(f"{label}: {exc!r}")
            print(label, "failed:", repr(exc))
    raise RuntimeError("Cannot load MindChat tokenizer. Attempts:\n" + "\n".join(errors))

tokenizer = load_mindchat_tokenizer(MODEL_NAME)

OFFLOAD_DIR = "/kaggle/working/mindchat_offload"
os.makedirs(OFFLOAD_DIR, exist_ok=True)

auth_kwargs = {"use_auth_token": HF_TOKEN} if HF_TOKEN else {}
base_kwargs = {
    "device_map": "auto",
    "trust_remote_code": True,
    "torch_dtype": torch.float16,
    "low_cpu_mem_usage": True,
    "offload_folder": OFFLOAD_DIR,
    **auth_kwargs,
}
if MAX_MEMORY_MAP is not None:
    # With two visible GPUs this becomes {0: "12GiB", 1: "12GiB", "cpu": "32GiB"}.
    base_kwargs["max_memory"] = MAX_MEMORY_MAP

def quant_4bit_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

def quant_8bit_config():
    return BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True,
    )

load_attempts = []
if torch.cuda.is_available():
    load_attempts.extend([
        ("AutoModelForCausalLM 4-bit", AutoModelForCausalLM, {**base_kwargs, "quantization_config": quant_4bit_config()}),
        ("AutoModel 4-bit", AutoModel, {**base_kwargs, "quantization_config": quant_4bit_config()}),
        ("AutoModelForCausalLM 8-bit CPU-offload", AutoModelForCausalLM, {**base_kwargs, "quantization_config": quant_8bit_config()}),
        ("AutoModel 8-bit CPU-offload", AutoModel, {**base_kwargs, "quantization_config": quant_8bit_config()}),
    ])

if ALLOW_FP16_FALLBACK or not torch.cuda.is_available():
    load_attempts.extend([
        ("AutoModelForCausalLM fp16 fallback", AutoModelForCausalLM, dict(base_kwargs)),
        ("AutoModel fp16 fallback", AutoModel, dict(base_kwargs)),
    ])
else:
    print("Skipping fp16 fallback on GPU. Set MINDCHAT_ALLOW_FP16_FALLBACK=1 only on larger GPUs.")

errors = []
model = None
for label, cls, kwargs in load_attempts:
    cleanup_cuda()
    try:
        print("Trying:", label)
        model = cls.from_pretrained(MODEL_NAME, **kwargs)
        print("Loaded with:", label)
        break
    except Exception as exc:
        message = f"{label}: {type(exc).__name__}: {exc}"
        errors.append(message)
        print(label, "failed:", repr(exc))
        try:
            del model
        except Exception:
            pass
        model = None
        cleanup_cuda()

if model is None:
    raise RuntimeError(
        "Cannot load MindChat model after all safe attempts.\n"
        "Restart runtime, run cells from top, and for dual T4 use MINDCHAT_GPU_MEMORY=12GiB,12GiB.\n"
        + "\n".join(errors)
    )

model.eval()
print("MindChat ready.")
print("Has chat method:", hasattr(model, "chat"))
print("hf_device_map:", getattr(model, "hf_device_map", None))


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'InternLMTokenizer'. 
The class this function is called from is 'LlamaTokenizer'.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message


Loading X-D-Lab/MindChat-7B...
CUDA GPUs visible at load: 2
GPU 0 free/total at load start: 14.46GB / 14.56GB
GPU 1 free/total at load start: 14.46GB / 14.56GB
Accelerate max_memory map: {0: '12GiB', 1: '12GiB', 'cpu': '32GiB'}
Trying tokenizer: LlamaTokenizer tokenizer.model
Tokenizer loaded with: LlamaTokenizer tokenizer.model
Skipping fp16 fallback on GPU. Set MINDCHAT_ALLOW_FP16_FALLBACK=1 only on larger GPUs.
Trying: AutoModelForCausalLM 4-bit


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded with: AutoModelForCausalLM 4-bit
MindChat ready.
Has chat method: True
hf_device_map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'lm_head': 1}


In [6]:
# 4. Standalone FastAPI server
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

app = FastAPI(title="MindChat Standalone Inference")


def parse_benchmark_prompt(prompt: str):
    """Parse prompt produced by backend/benchmarks/vsm/adapters/remote_chatbots.py.

    Expected shape:
      Lịch sử:\n...\nUser: current message\nMindChat:
    """
    prompt = prompt.strip()
    if "\nUser:" not in prompt:
        return prompt, []

    before_user, after_user = prompt.rsplit("\nUser:", 1)
    query = after_user.replace("MindChat:", "").strip()
    history_text = before_user.replace("Lịch sử:", "").strip()
    if history_text == "(trống)":
        history_text = ""

    history = []
    if history_text:
        chunks = re.split(r"\n(?=User:|Assistant:)", history_text)
        current_user = None
        for chunk in chunks:
            chunk = chunk.strip()
            if chunk.startswith("User:"):
                current_user = chunk[len("User:"):].strip()
            elif chunk.startswith("Assistant:") and current_user is not None:
                assistant = chunk[len("Assistant:"):].strip()
                history.append((current_user, assistant))
                current_user = None
    return query, history


def first_model_device():
    device_map = getattr(model, "hf_device_map", None)
    if isinstance(device_map, dict):
        cuda_devices = sorted({dev for dev in device_map.values() if isinstance(dev, int)})
        if cuda_devices:
            return torch.device(f"cuda:{cuda_devices[0]}")
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


@app.get("/health")
def health():
    return {
        "ok": True,
        "model": MODEL_NAME,
        "service": "mindchat",
        "cuda": torch.cuda.is_available(),
        "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "max_memory": MAX_MEMORY_MAP,
        "device_map": getattr(model, "hf_device_map", None),
        "has_chat": hasattr(model, "chat"),
    }


@app.post("/v1/generate")
async def generate(request: Request):
    try:
        data = await request.json()
        prompt = str(data.get("prompt", "")).strip()
        max_new_tokens = int(data.get("max_new_tokens", 512))
        temperature = float(data.get("temperature", 0.7))
        top_p = float(data.get("top_p", 0.9))
        repetition_penalty = float(data.get("repetition_penalty", 1.05))
        if not prompt:
            return JSONResponse({"text": ""})

        query, history = parse_benchmark_prompt(prompt)

        with torch.no_grad():
            if hasattr(model, "chat"):
                text, _ = model.chat(
                    tokenizer,
                    query=query,
                    history=history,
                    max_new_tokens=max_new_tokens,
                    do_sample=temperature > 0,
                    temperature=temperature,
                    top_p=top_p,
                    repetition_penalty=repetition_penalty,
                )
            else:
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
                inputs = {key: value.to(first_model_device()) for key, value in inputs.items()}
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=temperature > 0,
                    top_p=top_p,
                    repetition_penalty=repetition_penalty,
                    pad_token_id=tokenizer.eos_token_id,
                )
                generated = outputs[0][inputs["input_ids"].shape[-1]:]
                text = tokenizer.decode(generated, skip_special_tokens=True).strip()

        return JSONResponse({"text": str(text).strip(), "model": MODEL_NAME})
    except Exception as e:
        return JSONResponse({"error": repr(e)}, status_code=500)
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
# 5. Start ngrok + uvicorn
import uvicorn

# Do NOT paste the token here. In Kaggle, create a Secret named NGROK_AUTH_TOKEN.
NGROK_AUTH_TOKEN = get_secret_or_env("NGROK_AUTH_TOKEN")
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("WARNING: NGROK_AUTH_TOKEN is empty. Add it as Kaggle Secret named NGROK_AUTH_TOKEN.")

try:
    ngrok.kill()
except Exception as exc:
    print("ngrok.kill warning:", repr(exc))

public_url = ngrok.connect(8000, bind_tls=True).public_url
print("=" * 80)
print(f"MINDCHAT_NGROK_URL={public_url}")
print("Set this URL in backend .env, then benchmark will call only this MindChat server.")
print("Health:", f"{public_url}/health")
print("=" * 80)

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, loop="asyncio", log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [271]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


MINDCHAT_NGROK_URL=https://sprinkled-autopilot-disposal.ngrok-free.dev
Set this URL in backend .env, then benchmark will call only this MindChat server.
Health: https://sprinkled-autopilot-disposal.ngrok-free.dev/health
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "GET /health HTTP/1.1" 200 OK


2026-05-15 15:58:20.860961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778860701.099971     271 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778860701.167176     271 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778860701.728481     271 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778860701.728534     271 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778860701.728537     271 computation_placer.cc:177] computation placer alr

INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:4935:48d0:26ad:66a8:cf05:684b:0 - "POST /v1/generate HTTP/1.1" 200 OK
INFO:     2001:ee0:49